# C-15 Bias Mitigation Techniques

**Control:** C-15 Bias Mitigation Techniques, MAK-AI Responsible AI Toolkit.

**Learning goal:** apply one mitigation method before training (`CorrelationRemover`), one during training (`ExponentiatedGradient` with a fairness constraint) and one after training (`ThresholdOptimizer`); compare the trade-off between performance and the fairness metric you chose; and recognise when the right answer is better data or not deploying.

**Time to run:** about 3 minutes on an ordinary laptop. No GPU.

**Licence:** this notebook is Apache-2.0.

**Example data:** UCI Adult (Becker and Kohavi, 1996), DOI 10.24432/C5XW20, licence CC BY 4.0, 48,842 rows, from https://archive.ics.uci.edu/dataset/2/adult. It is credited here because CC BY 4.0 asks for attribution. The split and the baseline model are the same as in C-13 (same seed, same columns), so the numbers can be compared with that notebook.

**A caveat before you start:** Adult is a 1994 United States census extract with that year's categories for sex and race. The notebook teaches a method. It does not say these methods suit your data, and it does not say a model made fairer by them is fair.

## Linked risk areas and questions

Generated from content 5.0.0. Do not edit by hand.

Risk areas: 2

- IP-01: Algorithmic Discrimination
- IP-05: Model Robustness Failure

Questions: 13: Q-IP-02, Q-IP-04, Q-IP-05, Q-IP-06, Q-IP-08, Q-IP-17, Q-IP-18, Q-IP-13, Q-IP-15, Q-IP-19, Q-IP-22, Q-IP-24, Q-IP-EXTRA-01

## When to use this notebook

Use it when you have found a gap between groups (see C-13) and are answering questions about what you did about it, with status **Not yet** or **Partly**, especially with the reason code `ny_needs_skills` ("Needs skills or guidance the team does not have yet"). It helps with `Q-IP-15`: have you applied and tested a method to reduce bias, and what did it cost?

The summary this notebook writes can be attached as closure evidence of a mitigation recorded on the platform, marked "follows C-15".

You need a table with one row per person, the true outcome, the columns the model reads and the column that defines the groups.

In [ ]:
import hashlib
import importlib.metadata
import io
import platform
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from fairlearn.metrics import (
    demographic_parity_difference,
    equalized_odds_difference,
    plot_model_comparison,
    selection_rate,
)
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.preprocessing import CorrelationRemover
from fairlearn.reductions import DemographicParity, EqualizedOdds, ExponentiatedGradient
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 20261030  # the one random seed, the same as in C-13 so the split is the same
pd.set_option('display.width', 150)
pd.set_option('display.max_columns', 30)

PACKAGES = ['pandas', 'numpy', 'scikit-learn', 'fairlearn', 'matplotlib']

print('Python', platform.python_version())
for _package in PACKAGES:
    print(_package, importlib.metadata.version(_package))

In [ ]:
DATA_URL = 'https://archive.ics.uci.edu/static/public/2/data.csv'
DATA_SHA256 = '26b38f8623766396e2ba0d1a235a283db8ed993cc32bf32fc8aab09708c3bdf2'
DATA_FILE = 'uci-adult-2-data.csv'  # the cache file name is fixed here

CACHE_DIR = Path.home() / '.cache' / 'makrai-notebooks'


def fetch_checked(url, sha256, file_name):
    """Return the bytes of a dataset whose SHA-256 matches `sha256`.

    The file is downloaded once into the cache folder, and written there only after its hash
    matches. Every run, including the first, hashes the bytes before they are used, and a
    mismatch stops the notebook. `file_name` is fixed in this notebook, never taken from the address.
    """
    if not url.startswith('https://'):
        raise ValueError(f'the dataset address must be https: {url}')
    path = CACHE_DIR / file_name
    if path.exists():
        data, source, remedy = path.read_bytes(), f'the cached file {path}', f'Delete {path} and run again.'
    else:
        request = urllib.request.Request(url, headers={'User-Agent': 'makrai-notebooks'})
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                if not response.geturl().startswith('https://'):
                    raise RuntimeError(f'{url} redirected to a non-https address: {response.geturl()}')
                data = response.read()
        except OSError as error:
            raise RuntimeError(f'could not download {url}: {error}. If you cannot reach it, save the file by hand '
                               f'as {path}; it is checked against the pinned SHA-256 either way.') from error
        source, remedy = url, 'Do not use this download; check the dataset page for a new version.'
    found = hashlib.sha256(data).hexdigest()
    if found != sha256:
        raise RuntimeError(f'{file_name}: the SHA-256 of {source} is {found}, expected {sha256}. {remedy}')
    if not path.exists():
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        partial = path.with_name(path.name + '.part')
        partial.write_bytes(data)
        partial.replace(path)  # the cache never holds a half-written file
    return data


# To use your own file, replace the next line with:  raw = Path('your-file.csv').read_bytes()
raw = fetch_checked(DATA_URL, DATA_SHA256, DATA_FILE)
DATA_SHA256_USED = hashlib.sha256(raw).hexdigest()  # the evidence summary records the hash of the bytes used
df = pd.read_csv(io.BytesIO(raw))
df['income'] = df['income'].str.rstrip('.')  # Adult only: the census test file writes the label with a trailing dot
print(f'{len(df):,} rows, {df.shape[1]} columns; SHA-256 of the data used: {DATA_SHA256_USED[:12]}...')

In [ ]:
# ADAPT HERE. Edit only this cell to run the notebook on your own data. Every later cell reads only these names.
LABEL = 'income'                 # the column that holds the true outcome
FAVOURABLE = '>50K'              # the value of LABEL that counts as the good outcome for a person
SENSITIVE = 'sex'                # the column that defines the groups the methods try to treat alike
NUMERIC = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
CATEGORICAL = ['workclass', 'marital-status', 'occupation', 'relationship']
USE_EQUALIZED_ODDS = False       # False asks for equal selection rates, True asks for equal error rates
REMOVER_STRENGTH = 1.0           # 1.0 removes the linear link to the group columns fully, 0.0 not at all
TOLERANCE = 0.01                 # how far from perfectly equal the in-training method is allowed to settle

### Step 1: the baseline, the same as C-13

**What this does.** It splits the data exactly as C-13 does (70 percent to train, 30 percent to test, same seed), turns the columns into numbers, and fits a plain logistic regression that never sees the group column. It then reports how accurate it is and how large the gaps between groups are, using two summary numbers from Fairlearn:

- **demographic parity difference**: the gap between the group with the highest selection rate (share labelled favourable) and the lowest. Zero means equal rates.
- **equalised odds difference**: the larger of the gap in the true positive rate and the gap in the false positive rate. Zero means both errors are shared evenly.

Every model below is judged on the same test people with the same two numbers, plus accuracy.

**What it shows.** The starting point. Any method has to be compared with it.

**What it does not show.** Whether the baseline gap is a problem, or the right one to reduce. That choice comes before this notebook (see C-13).

In [ ]:
y = (df[LABEL] == FAVOURABLE).astype(int)
train_index, test_index = train_test_split(df.index, test_size=0.3, stratify=y, random_state=SEED)

preprocess = ColumnTransformer(
    [('numbers', StandardScaler(), NUMERIC), ('categories', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), CATEGORICAL)],
    verbose_feature_names_out=False).set_output(transform='pandas')
inputs = df[NUMERIC + CATEGORICAL].copy()
inputs[CATEGORICAL] = inputs[CATEGORICAL].fillna('missing').astype(str)
M_train = preprocess.fit_transform(inputs.loc[train_index])
M_test = preprocess.transform(inputs.loc[test_index])
group_train = df.loc[train_index, SENSITIVE].astype(str)
group_test = df.loc[test_index, SENSITIVE].astype(str)
y_train, y_test = y[train_index], y[test_index]

fairness_metric = equalized_odds_difference if USE_EQUALIZED_ODDS else demographic_parity_difference
constraint_name = 'equalized_odds' if USE_EQUALIZED_ODDS else 'demographic_parity'
predictions, rows = {}, []


def record(name, y_pred):
    predictions[name] = np.asarray(y_pred)
    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'balanced accuracy': balanced_accuracy_score(y_test, y_pred),
        'selection rate': selection_rate(y_test, y_pred),
        'demographic parity difference': demographic_parity_difference(y_test, y_pred, sensitive_features=group_test),
        'equalized odds difference': equalized_odds_difference(y_test, y_pred, sensitive_features=group_test),
    })


baseline = LogisticRegression(max_iter=1000).fit(M_train, y_train)
record('baseline', baseline.predict(M_test))
print(f'metric the methods try to improve: {fairness_metric.__name__}')
print(pd.DataFrame(rows).round(3).to_string(index=False))

### Step 2: before training, `CorrelationRemover`

**What this does.** It changes the **inputs** before any model sees them. For each column it removes the part that can be predicted linearly from the group columns, so that what is left is, linearly, unrelated to group. `REMOVER_STRENGTH` between 0 and 1 removes only part of it. The group columns are then dropped from the inputs and the same logistic regression is fitted on the cleaned inputs.

**What it shows.** What happens to accuracy and to the gaps when the inputs carry less linear information about group.

**What it does not show.** That the group is gone. Only the linear link is removed: a non-linear pattern, or a combination of columns, can still reveal it. It also changes the meaning of every column, so the model's inputs are harder to explain.

In [ ]:
group_levels = sorted(df[SENSITIVE].dropna().astype(str).unique())
sensitive_names = [f'sensitive={level}' for level in group_levels]


def with_group_columns(matrix, groups):
    extra = pd.DataFrame({name: (groups.to_numpy() == level).astype(float) for name, level in zip(sensitive_names, group_levels)},
                         index=matrix.index)
    return pd.concat([matrix, extra], axis=1)


remover = CorrelationRemover(sensitive_feature_ids=sensitive_names, alpha=REMOVER_STRENGTH)
R_train = remover.fit_transform(with_group_columns(M_train, group_train))
R_test = remover.transform(with_group_columns(M_test, group_test))
remover_model = LogisticRegression(max_iter=1000).fit(R_train, y_train)
record('before training: CorrelationRemover', remover_model.predict(R_test))
print(pd.DataFrame(rows).round(3).to_string(index=False))

### Step 3: during training, `ExponentiatedGradient`

**What this does.** It trains the model **with the fairness rule inside the training**. Fairlearn fits a series of logistic regressions, each time giving more weight to the people the last model treated worst on the chosen rule (equal selection rates, or equal error rates), and combines them into one randomised model. `TOLERANCE` says how far from perfectly equal it may settle. Predictions use a seed, because the combined model flips a weighted coin between its parts.

**What it shows.** What accuracy costs when the rule is part of the objective.

**What it does not show.** That the rule was the right one. And the result is a mix of models with a random choice at prediction time, which can be harder to explain and to audit: the same person can get different answers on different runs unless the seed is fixed.

In [ ]:
rule = EqualizedOdds() if USE_EQUALIZED_ODDS else DemographicParity()
mitigator = ExponentiatedGradient(LogisticRegression(max_iter=1000), constraints=rule, eps=TOLERANCE)
mitigator.fit(M_train, y_train, sensitive_features=group_train)
record('during training: ExponentiatedGradient', mitigator.predict(M_test, random_state=SEED))
print(pd.DataFrame(rows).round(3).to_string(index=False))

### Step 4: after training, `ThresholdOptimizer`

**What this does.** It leaves the trained model alone and changes only the **decision threshold**, separately for each group, to meet the same rule while losing as little accuracy as it can. Where one threshold per group cannot hit the rule exactly, it flips a weighted coin between two thresholds.

**What it shows.** The cheapest of the three to try, because it needs no retraining. Often it closes the gap with a small loss in accuracy.

**What it does not show.** Two things you must decide before using it. First, it **needs the group of each person when the model is used**, not only in training. Whether you may collect, store and use that, and whether using it is lawful and acceptable where you work, is itself a question for deployment. Second, it treats people differently by group on purpose: two people with the same score can get different decisions. That can be what you want, and it can be what a law or a community forbids.

In [ ]:
optimizer = ThresholdOptimizer(estimator=baseline, constraints=constraint_name, objective='accuracy_score',
                               prefit=True, predict_method='predict_proba')
optimizer.fit(M_train, y_train, sensitive_features=group_train)
record('after training: ThresholdOptimizer', optimizer.predict(M_test, sensitive_features=group_test, random_state=SEED))
comparison = pd.DataFrame(rows).set_index('model')
comparison['accuracy lost vs baseline'] = comparison.loc['baseline', 'accuracy'] - comparison['accuracy']
print(comparison.round(3).to_string())

### Step 5: the trade-off, side by side

**What this does.** It plots each model by its accuracy (across) and the gap you chose to reduce (up), using Fairlearn's `plot_model_comparison`. A model further left and lower has traded accuracy for a smaller gap. The table above has the numbers.

**What it shows.** The price of each method, in the units you chose, on this data and this test set. Read the other gap in the table as well as the one you chose: with the equal-selection rule on this data, the error-rate gap grows as the selection-rate gap shrinks. That is the conflict between fairness definitions that C-13 explains, seen in practice.

**What it does not show.** Whether a price is acceptable (that is the "Stop and decide" cell), or what the trade-off would be on different data. One test set gives one picture, and the differences between methods may be smaller than the noise (see the intervals in C-13 and C-14).

In [ ]:
fig, axis = plt.subplots(figsize=(7, 5))
plot_model_comparison(y_preds=predictions, y_true=y_test, sensitive_features=group_test,
                      x_axis_metric=accuracy_score, y_axis_metric=fairness_metric,
                      ax=axis, point_labels=True, legend=False)
axis.set_title('Accuracy against the gap between groups')
fig.tight_layout()

### Step 6: when the answer is better data, or not deploying

**What this does.** No code, only a prompt, because no method on this page is always the answer.

- A gap that comes from **missing or unrepresentative data** (a group with few examples, labels that were made differently for one group) is best fixed at the source: collect more, collect better, relabel. A mitigation method can hide it.
- A gap that **no method closes without a loss you cannot accept** is a result. It may mean the model should not be used for this decision, or only with a person reviewing every case.
- A method that **removes a gap in the test data** can fail to do so in use, if the people or the world differ from the test set (see C-21).
- Fairlearn has a fourth family, adversarial methods (`AdversarialFairnessClassifier`), that this notebook does not cover because they need PyTorch or TensorFlow. It is named here as further reading.

**What it shows.** That mitigation is one option among several, and not always the first.

**What it does not show.** Which option fits your case. That is the team's, and the "Stop and decide" cell records it.

### Stop and decide: which trade-off is acceptable, and who decides

The notebook shows what each method costs. It cannot tell you what you are willing to pay.

Questions to ask in the room, including people who speak for those affected:

1. How much accuracy are you willing to give up to shrink the gap, and who pays for the extra errors? Are the extra errors spread the way you intended?
2. The metric you chose here (equal selection rates, or equal error rates): is it the one that fits the harm? See C-13.
3. Can you collect and use the group of each person when the model is used? Would you be allowed to? Do the people affected accept being treated differently by group, if `ThresholdOptimizer` is the choice?
4. Would better data or a different design do more than any of these methods?
5. Should the model be used at all for this decision, or only with a person reviewing it?
6. Who decides, where is it written, and when will you check again?

Write your choices in the next cell. Leave it blank if you have not decided, and the summary will say so.

In [ ]:
# YOUR CHOICES. The notebook does not fill this in. Plain words are fine.
METHOD_WE_CHOSE = ''            # which method, or none, and why
TRADE_OFF_WE_ACCEPT = ''        # the accuracy and gap numbers you accept, in your words
WHO_DECIDED = ''                # the role or body that decided, and where it is written
ALTERNATIVES_CONSIDERED = ''    # better data, human review, not deploying: what you weighed
NEXT_CHECK = ''                 # when you will measure again, and on what data

In [ ]:
EVIDENCE_DIR = Path('evidence')
EVIDENCE_DIR.mkdir(exist_ok=True)

# A small table of numbers, one row per model. It holds no row of the data.
comparison.round(4).to_csv(EVIDENCE_DIR / 'C-15_comparison.csv')
fig.savefig(EVIDENCE_DIR / 'C-15_trade_off.png', dpi=100, bbox_inches='tight')

versions = '\n'.join(f'- {package} {importlib.metadata.version(package)}' for package in PACKAGES)
summary = f"""# C-15 Bias Mitigation Techniques: evidence summary

Attach this file, with the CSV and PNG files beside it, as closure evidence of a mitigation marked "follows C-15" (question Q-IP-15). It holds numbers only, never the data.

## What was run
- Outcome: {LABEL}, favourable value {FAVOURABLE}; {len(df):,} rows, {len(y_test):,} in the test set (the same split as C-13)
- Groups: {SENSITIVE}
- Rule the methods tried to meet: {constraint_name}; gap reported: {fairness_metric.__name__}
- Methods: CorrelationRemover (strength {REMOVER_STRENGTH}), ExponentiatedGradient (tolerance {TOLERANCE}), ThresholdOptimizer (needs the group of each person when the model is used)

## Results on the test set
{comparison.round(3).to_string()}

## Choices made in the stop-and-decide cell
- Method chosen: {METHOD_WE_CHOSE or 'not written yet'}
- Trade-off accepted: {TRADE_OFF_WE_ACCEPT or 'not written yet'}
- Who decided: {WHO_DECIDED or 'not written yet'}
- Alternatives considered: {ALTERNATIVES_CONSIDERED or 'not written yet'}
- Next check: {NEXT_CHECK or 'not written yet'}

## Files beside this summary
C-15_comparison.csv, C-15_trade_off.png

## Package versions
- Python {platform.python_version()}
{versions}

## Data
SHA-256 of the data file used: {DATA_SHA256_USED}
"""
(EVIDENCE_DIR / 'C-15_summary.md').write_text(summary, encoding='utf-8')
print(summary)

## What this does not show, and where to go next

- **Reducing a gap is not making the system fair.** Each method moves one number. Harms that the number does not measure stay.
- **One test set, one picture.** The differences between methods can be inside the noise. Check with intervals before choosing between close results.
- **Methods that use the group at prediction time need that data, and the right to use it.** `ThresholdOptimizer` is the example here.
- **Mitigation can hide a data problem.** If a group is badly represented or mislabelled, fix the data first.
- **Adversarial debiasing is not covered.** Fairlearn's `AdversarialFairnessClassifier` needs PyTorch or TensorFlow and is named as further reading.

Where to go next: C-13 Subgroup Fairness Metrics (choosing the metric), C-14 Disaggregated Performance Analysis (intervals and small groups), C-21 Monitoring and Drift Detection (does the gap stay closed), and the model card template, whose mitigation section takes this summary.